# 04 — Training DenseNet121 (5-Fold Cross Validation)

**Skripsi:** Klasifikasi Tumor Paru pada Citra CT Scan Menggunakan DenseNet121
dan Explainable Artificial Intelligence berbasis Grad-CAM.

| Aspek | Nilai |
|---|---|
| Model | `DenseNet121(weights="imagenet")` |
| Input | `(224, 224, 3)` — **sudah** diproses `densenet.preprocess_input` |
| Klasifikasi | binary: Benign (0) vs Malignant (1), aktivasi `sigmoid` |
| Validasi | 5-Fold, grouping `PatientID` (dari notebook 02) |
| Device | **GPU WSL2 wajib** — tanpa CPU fallback |

### ⚠️ Kenapa notebook ini harus jalan di WSL2

TensorFlow 2.21 **tidak pernah** memakai GPU di native Windows (sejak TF 2.11).
Training di sini **tidak bisa jalan di Windows**. Cell 1 memverifikasi GPU dan
**berhenti dengan error** bila GPU tidak terdeteksi — tidak ada fallback ke CPU.

### ⚠️ Data sudah diproses

File `.npy` di `dataset/04_preprocessed/` **sudah** melewati
`tf.keras.applications.densenet.preprocess_input`. Notebook ini **tidak**
memanggil `preprocess_input` lagi, dan **tidak** menormalisasi ulang.

### Batas kerja notebook ini

Notebook ini fokus pada **training**. Tidak mencakup confusion matrix,
precision/recall/F1/ROC-AUC final, maupun Grad-CAM — itu tahap berikutnya.

## 1. Validasi Environment + GPU (WAJIB)

Cell ini **berhenti dengan error** bila GPU tidak terdeteksi. Training di CPU
dilarang karena RTX 4070 Anda tidak akan terpakai.

In [32]:
# --- CELL 1 - WAJIB GPU --------------------------------------------------
import os

# TensorFlow harus tahu GPU SEBELUM diinisialisasi context.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import sklearn

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("=" * 74)
print("VALIDASI ENVIRONMENT")
print("=" * 74)
print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {sys.platform}")
print(f"TensorFlow   : {tf.__version__}")
print(f"Keras        : {keras.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"Random seed  : {RANDOM_SEED}")

print()
print("=" * 74)
print("VALIDASI GPU")
print("=" * 74)
print("tf.config.list_physical_devices('GPU') :")
gpus = tf.config.list_physical_devices("GPU")
print(f"  {gpus}")
print(f"Jumlah GPU : {len(gpus)}")

if not gpus:
    raise RuntimeError(
        "GPU tidak terdeteksi. Training dihentikan untuk mencegah training "
        "bernjalan di CPU.\n\n"
        "TensorFlow 2.21 tidak mendukung GPU di native Windows. "
        "Jalankan notebook ini di WSL2 dengan:\n"
        "    pip install tensorflow[and-cuda]==2.21.0\n\n"
        "Detail device yang terdeteksi TensorFlow: "
        f"{tf.config.list_physical_devices()}"
    )

# memory growth supaya alokasi tidak langsung memakan semua VRAM
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
        print(f"  memory growth aktif : {g.name}")
    except RuntimeError as e:
        print(f"  memory growth GAGAL di {g.name}: {e}")

for g in gpus:
    print(f"\n  GPU dipakai: {g.name}")
    for d in tf.config.experimental.get_device_details(g):
        print(f"    {d}")
print(f"\nLogical GPU: {tf.config.list_logical_devices('GPU')}")

# konfirmasi GPU benar-benar dipakai (bukan hanya terdaftar)
# Catatan: gpus[0].name = '/physical_device:GPU:0' TIDAK valid sebagai
# device spec untuk tf.device(). Gunakan '/GPU:0'.
_probe = tf.zeros((8, 224, 224, 3), dtype=tf.float32)
with tf.device("/GPU:0"):
    _out = tf.reduce_sum(_probe)
    _placed = _out.device
print(f"\nSmoke test pada /GPU:0 : sum = {float(_out):.1f}  device = {_placed}")
if "GPU" not in _placed:
    raise RuntimeError(
        f"Operasi tidak berjalan di GPU (device = {_placed}). "
        f"Training dihentikan."
    )
print("GPU terkonfirmasi dipakai untuk komputasi.")


def gpu_mem_gb():
    """(allocated_GB, peak_GB). Spec valid: 'GPU:0' (bukan 'physical_device:...')."""
    info = tf.config.experimental.get_memory_info("GPU:0")
    return info["current"] / 1024**3, info["peak"] / 1024**3


VALIDASI ENVIRONMENT
Python       : 3.12.13
Platform     : linux
TensorFlow   : 2.21.0
Keras        : 3.15.1
NumPy        : 2.2.6
pandas       : 2.2.3
scikit-learn : 1.6.1
Random seed  : 42

VALIDASI GPU
tf.config.list_physical_devices('GPU') :
  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Jumlah GPU : 1
  memory growth aktif : /physical_device:GPU:0

  GPU dipakai: /physical_device:GPU:0
    compute_capability
    device_name

Logical GPU: [LogicalDevice(name='/device:GPU:0', device_type='GPU')]

Smoke test pada /GPU:0 : sum = 0.0  device = /job:localhost/replica:0/task:0/device:GPU:0
GPU terkonfirmasi dipakai untuk komputasi.


## 2. Konfigurasi Training

Semua path relatif terhadap `PROJECT_ROOT` agar portabel.

In [33]:
# --- CELL 2 - KONFIGURASI & PATH ------------------------------------------
MANIFEST_RELPATH = Path("dataset/04_preprocessed/preprocessed_manifest.csv")
MODELS_RELDIR = Path("models/densenet121")
RESULTS_RELDIR = Path("results/training")

N_SPLITS = 5
TARGET_SIZE = (224, 224)
NUM_CHANNELS = 3
INPUT_SHAPE = (224, 224, 3)

# Data SUDAH diproses densenet.preprocess_input -> JANGAN preprocess lagi
ALREADY_PREPROCESSED = True

# --- HYPERPARAMETER (disetujui penulis skripsi) --------------------------
BATCH_SIZE = 32              # realistis untuk RTX 4070 12 GB
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 20
STAGE1_LR = 1e-3
STAGE2_LR = 1e-5
OPTIMIZER = "adam"
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MONITOR = "val_loss"
LR_PLATEAU_PATIENCE = 3
LR_PLATEAU_FACTOR = 0.5
FINE_TUNE_LAST_N_LAYERS = 30
LOSS = "binary_crossentropy"

# Augmentation: TRAINING SAJA (disetujui)
AUGMENT_ENABLED = True
AUG_FLIP_HORIZONTAL = True
AUG_ROTATION_DEG = 10.0
AUG_ZOOM_RANGE = (-0.1, 0.1)
# TIDAK ADA vertical flip / shear / elastic


def find_project_root(marker: Path, start: Path | None = None) -> Path:
    base = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [base, *base.parents]:
        if (candidate / marker).exists():
            return candidate
    searched = "\n".join(f"  - {p}" for p in [base, *base.parents])
    raise FileNotFoundError(
        f"PROJECT_ROOT tidak ditemukan.\n"
        f"Notebook mencari: {marker}\n\n"
        f"Direktori yang diperiksa:\n{searched}"
    )


PROJECT_ROOT = find_project_root(MANIFEST_RELPATH)
MANIFEST_FILE = PROJECT_ROOT / MANIFEST_RELPATH
MODELS_DIR = PROJECT_ROOT / MODELS_RELDIR
RESULTS_DIR = PROJECT_ROOT / RESULTS_RELDIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("KONFIGURASI TRAINING")
print("-" * 74)
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"Manifest       : {MANIFEST_RELPATH}")
print(f"Model output   : {MODELS_RELDIR}")
print(f"History output : {RESULTS_RELDIR}")
print(f"Folds          : {N_SPLITS}")
print(f"Input shape    : {INPUT_SHAPE}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Data processed : {ALREADY_PREPROCESSED} "
      f"(JANGAN panggil preprocess_input lagi)")


KONFIGURASI TRAINING
--------------------------------------------------------------------------
PROJECT_ROOT   : /mnt/c/230016_rivan/source-code2020
Manifest       : dataset/04_preprocessed/preprocessed_manifest.csv
Model output   : models/densenet121
History output : results/training
Folds          : 5
Input shape    : (224, 224, 3)
Batch size     : 32
Data processed : True (JANGAN panggil preprocess_input lagi)


## 3. Validasi Dataset

Memuat `preprocessed_manifest.csv` dan memvalidasi seluruh file `.npy`.
**Split tidak dibuat ulang** — `Fold` dari notebook 02 dipakai apa adanya.

In [34]:
# --- CELL 3 - LOAD + VALIDASI DATASET ------------------------------------
man = pd.read_csv(MANIFEST_FILE, dtype={
    "PatientID": str, "SeriesInstanceUID": str, "NoduleClusterID": str})

print("VALIDASI DATASET")
print("=" * 74)
print(f"baris manifest      : {len(man)}")
print(f"Benign (LabelBinary 0): {int((man.LabelBinary == 0).sum())}")
print(f"Malignant (1)         : {int((man.LabelBinary == 1).sum())}")
print(f"PatientID unik        : {man.PatientID.nunique()}")
print(f"SeriesInstanceUID     : {man.SeriesInstanceUID.nunique()}")
print(f"Fold                  : {sorted(man.Fold.unique().tolist())}")
print(f"sample per fold       : "
      f"{man.Fold.value_counts().sort_index().to_dict()}")

# ---cek file ada, shape, dtype, NaN, Inf ---------------------------------
bad = {"missing": [], "shape": [], "dtype": [], "nan": [], "inf": []}
for _, r in man.iterrows():
    p = PROJECT_ROOT / r.PreprocessedPath
    if not p.is_file():
        bad["missing"].append(str(r.NoduleClusterID))
        continue
    arr = np.load(p, mmap_mode="r")
    if arr.shape != INPUT_SHAPE:
        bad["shape"].append((str(r.NoduleClusterID), arr.shape))
    if arr.dtype != np.float32:
        bad["dtype"].append((str(r.NoduleClusterID), str(arr.dtype)))
    if np.isnan(np.asarray(arr)).any():
        bad["nan"].append(str(r.NoduleClusterID))
    if np.isinf(np.asarray(arr)).any():
        bad["inf"].append(str(r.NoduleClusterID))

print()
for k, v in bad.items():
    print(f"  {k:<10} bermasalah : {len(v)} {v[:3] if v else ''}")

problems = {k: v for k, v in bad.items() if v}
if problems:
    raise ValueError(f"Dataset bermasalah: {problems}. "
                     f"PROSES DIHENTIKAN. Sample tidak dibuang.")

assert len(man) == 524, "harus 524 sample"
assert set(man.LabelBinary.unique()) == {0, 1}, "LabelBinary harus 0/1"
assert set(man.Fold.unique()) <= {1, 2, 3, 4, 5}, "Fold harus 1..5"
print("\nSeluruh file valid: shape (224,224,3) float32, tanpa NaN/Inf.")


VALIDASI DATASET
baris manifest      : 524
Benign (LabelBinary 0): 381
Malignant (1)         : 143
PatientID unik        : 225
SeriesInstanceUID     : 225
Fold                  : [1, 2, 3, 4, 5]
sample per fold       : {1: 82, 2: 126, 3: 106, 4: 97, 5: 113}

  missing    bermasalah : 0 
  shape      bermasalah : 0 
  dtype      bermasalah : 0 
  nan        bermasalah : 0 
  inf        bermasalah : 0 

Seluruh file valid: shape (224,224,3) float32, tanpa NaN/Inf.


## 4. Validasi Data Leakage

Untuk fold ke-`k`: **train** = `Fold != k`, **validation** = `Fold == k`.

`set(train PatientID) ∩ set(validation PatientID)` harus kosong.

In [35]:
# --- CELL 4 - LEAKAGE PER FOLD ------------------------------------------
leak = []
print("VALIDASI LEAKAGE")
print("=" * 74)
for k in range(1, N_SPLITS + 1):
    tr = man[man.Fold != k]
    va = man[man.Fold == k]
    tr_pat = set(tr.PatientID)
    va_pat = set(va.PatientID)
    ov = tr_pat & va_pat
    print(f"  Fold {k}: train {len(tr):>3} sample / {len(tr_pat):>3} pasien | "
          f"val {len(va):>3} sample / {len(va_pat):>3} pasien | "
          f"overlap {len(ov)}")
    if ov:
        leak.append((k, len(ov)))
    n_multi = int((tr.groupby("PatientID").Fold.nunique() > 1).sum())
    if n_multi:
        leak.append((k, f"{n_multi} patient multi-fold"))

if leak:
    raise ValueError(f"Leakage terdeteksi: {leak}. PROSES DIHENTIKAN.")
print("\nTidak ada leakage. Grouping PatientID dipertahankan.")
print("Tidak ada train_test_split acak di notebook ini.")


VALIDASI LEAKAGE
  Fold 1: train 442 sample / 185 pasien | val  82 sample /  40 pasien | overlap 0
  Fold 2: train 398 sample / 178 pasien | val 126 sample /  47 pasien | overlap 0
  Fold 3: train 418 sample / 179 pasien | val 106 sample /  46 pasien | overlap 0
  Fold 4: train 427 sample / 180 pasien | val  97 sample /  45 pasien | overlap 0
  Fold 5: train 411 sample / 178 pasien | val 113 sample /  47 pasien | overlap 0

Tidak ada leakage. Grouping PatientID dipertahankan.
Tidak ada train_test_split acak di notebook ini.


## 5. Class Weight (per Training Fold)

Formula **balanced**, dihitung **hanya dari training fold**:

```
weight_c = N_train / (2 * N_class_train)
```

Dihitung ulang untuk setiap fold. Tidak pernah dari validation fold atau
seluruh dataset. Tidak ada oversampling maupun undersampling.

In [36]:
# --- CELL 5 - CLASS WEIGHT PER FOLD --------------------------------------
def balanced_class_weight(train_df: pd.DataFrame) -> dict[int, float]:
    """weight_c = N_train / (2 * N_class_train). Hanya dari training fold."""
    n_train = len(train_df)
    counts = train_df.LabelBinary.value_counts().to_dict()
    if set(counts) != {0, 1}:
        raise ValueError(f"Training fold harus punya kedua kelas, dapat {counts}")
    return {c: n_train / (2.0 * counts[c]) for c in (0, 1)}


CLASS_WEIGHTS = {}
print("CLASS WEIGHT PER TRAINING FOLD (balanced)")
print("=" * 74)
print(f"{'Fold':>4} {'N_train':>8} {'N_val':>6} {'train_0':>8} {'train_1':>8} "
      f"{'w0':>8} {'w1':>8}")
for k in range(1, N_SPLITS + 1):
    tr = man[man.Fold != k]
    va = man[man.Fold == k]
    cw = balanced_class_weight(tr)
    CLASS_WEIGHTS[k] = cw
    n0 = int((tr.LabelBinary == 0).sum())
    n1 = int((tr.LabelBinary == 1).sum())
    print(f"{k:>4} {len(tr):>8} {len(va):>6} {n0:>8} {n1:>8} "
          f"{cw[0]:>8.4f} {cw[1]:>8.4f}")

print("\nVerifikasi formula (w1/w0 harus = n0/n1):")
for k in range(1, N_SPLITS + 1):
    tr = man[man.Fold != k]
    n0 = int((tr.LabelBinary == 0).sum())
    n1 = int((tr.LabelBinary == 1).sum())
    ratio = CLASS_WEIGHTS[k][1] / CLASS_WEIGHTS[k][0]
    assert abs(ratio - n0 / n1) < 1e-3, f"Fold {k} tidak konsisten"
    print(f"  Fold {k}: w1/w0 = {ratio:.4f}  vs  n0/n1 = {n0 / n1:.4f}  [OK]")

print("\nClass weight TIDAK di-hard-code; dihitung per training fold.")
print("Tidak ada oversampling / undersampling.")


CLASS WEIGHT PER TRAINING FOLD (balanced)
Fold  N_train  N_val  train_0  train_1       w0       w1
   1      442     82      324      118   0.6821   1.8729
   2      398    126      287      111   0.6934   1.7928
   3      418    106      307      111   0.6808   1.8829
   4      427     97      312      115   0.6843   1.8565
   5      411    113      294      117   0.6990   1.7564

Verifikasi formula (w1/w0 harus = n0/n1):
  Fold 1: w1/w0 = 2.7458  vs  n0/n1 = 2.7458  [OK]
  Fold 2: w1/w0 = 2.5856  vs  n0/n1 = 2.5856  [OK]
  Fold 3: w1/w0 = 2.7658  vs  n0/n1 = 2.7658  [OK]
  Fold 4: w1/w0 = 2.7130  vs  n0/n1 = 2.7130  [OK]
  Fold 5: w1/w0 = 2.5128  vs  n0/n1 = 2.5128  [OK]

Class weight TIDAK di-hard-code; dihitung per training fold.
Tidak ada oversampling / undersampling.


## 6. Data Loading (`tf.data`)

File `.npy` dimuat **on-demand per batch** — 301 MB total tidak pernah dimuat
sekaligus. Augmentation diterapkan **on-the-fly, training saja**.

In [37]:
# --- CELL 6 - TF.DATA PIPELINE -------------------------------------------
from functools import partial

AUT = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(AUG_ROTATION_DEG / 180.0),
    tf.keras.layers.RandomZoom(AUG_ZOOM_RANGE[0], AUG_ZOOM_RANGE[1]),
], name="train_augmentation")

AUT_VIS = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(AUG_ROTATION_DEG / 180.0),
    tf.keras.layers.RandomZoom(AUG_ZOOM_RANGE[0], AUG_ZOOM_RANGE[1]),
], name="val_augmentation_DISABLED")


def _load_npy_impl(path_bytes) -> np.ndarray:
    """Dipanggil di dalam tf.py_function. Path datang sebagai bytes tensor."""
    s = path_bytes.numpy().decode("utf-8")
    return np.asarray(np.load(PROJECT_ROOT / s), dtype=np.float32)


def load_npy(path_tensor, label):
    """Baca .npy di dalam pipeline tf.data, TANPA memuat seluruh dataset.

    `np.load` tidak menerima tf.string/bytes tensor, jadi pembacaan
    dilakukan lewat tf.py_function yang menjembatani python ke TensorFlow.
    Output di-declare float32 (224,224,3) supaya shape-nya statis.
    """
    x = tf.py_function(_load_npy_impl, [path_tensor], tf.float32)
    x.set_shape(INPUT_SHAPE)
    return x, label


def make_dataset(sub: pd.DataFrame, batch_size: int, training: bool,
                 shuffle: bool = True) -> tf.data.Dataset:
    paths = [str(p) for p in sub.PreprocessedPath]
    labels = sub.LabelBinary.to_numpy().astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_npy, num_parallel_calls=tf.data.AUTOTUNE,
                deterministic=not training)

    if training and AUGMENT_ENABLED:
        # augmentation ON-THE-FLY, hanya training
        ds = ds.map(lambda x, y: (AUT(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    # validation: TIDAK ada augmentation sama sekali

    if shuffle:
        ds = ds.shuffle(min(len(sub), 4096), seed=RANDOM_SEED,
                        reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


print("PIPELINE tfn")
print("-" * 74)
print(f"  batch size       : {BATCH_SIZE}")
print(f"  prefetch         : AUTOTUNE")
print(f"  augmentation     : {'AKTIF (training only)' if AUGMENT_ENABLED else 'OFF'}")
print(f"    flip horizontal: {AUG_FLIP_HORIZONTAL}")
print(f"    rotation       : +/- {AUG_ROTATION_DEG} deg")
print(f"    zoom           : {AUG_ZOOM_RANGE}")
print(f"    vertical flip  : TIDAK")
print(f"    shear          : TIDAK")
print(f"    elastic        : TIDAK")
print(f"  validation aug   : TIDAK ADA")

# smoke test
_tr = man[man.Fold != 1]
ds = make_dataset(_tr.head(64), BATCH_SIZE, training=True)
xb, yb = next(iter(ds))
print(f"\n  smoke test batch: X {xb.shape} {xb.dtype} | y {yb.shape} {yb.dtype}")
print(f"  X finite         : {bool(np.isfinite(xb.numpy()).all())}")
print(f"  y nilai          : {sorted(set(yb.numpy().tolist()))}")
print(f"  X rentang        : {xb.numpy().min():.3f} .. {xb.numpy().max():.3f}")

_v = make_dataset(man[man.Fold == 1].head(64), BATCH_SIZE, training=False)
xv, yv = next(iter(_v))
print(f"  val batch        : X {xv.shape} | y {sorted(set(yv.numpy().tolist()))}")
print("\nDataset dibangun. Tidak ada preprocess_input di pipeline ini.")


PIPELINE tfn
--------------------------------------------------------------------------
  batch size       : 32
  prefetch         : AUTOTUNE
  augmentation     : AKTIF (training only)
    flip horizontal: True
    rotation       : +/- 10.0 deg
    zoom           : (-0.1, 0.1)
    vertical flip  : TIDAK
    shear          : TIDAK
    elastic        : TIDAK
  validation aug   : TIDAK ADA

  smoke test batch: X (32, 224, 224, 3) <dtype: 'float32'> | y (32,) <dtype: 'float32'>
  X finite         : True
  y nilai          : [0.0, 1.0]
  X rentang        : -1.169 .. 2.640
  val batch        : X (32, 224, 224, 3) | y [0.0, 1.0]

Dataset dibangun. Tidak ada preprocess_input di pipeline ini.


## 7. Model

```
DenseNet121(include_top=False, weights="imagenet", input_shape=(224,224,3))
  -> GlobalAveragePooling2D
  -> Dense(1, activation="sigmoid")
```

Loss `binary_crossentropy`. Output sigmoid = probabilitas Malignant.

**Penting:** `base.trainable = False` di-set **sebelum** backbone dimasukkan ke
`Sequential`. Kalau tidak, seluruh backbone ikut ter-latih.

In [38]:
# --- CELL 7 - BUILD MODEL ------------------------------------------------
def build_model(fine_tune_last_n: int = 0) -> tf.keras.Model:
    """Bangun DenseNet121 + head.

    fine_tune_last_n = 0  -> Stage 1, hanya head yang dilatih
    fine_tune_last_n > 0  -> Stage 2, buka N layer terakhir
    """
    base = tf.keras.applications.DenseNet121(
        include_top=False, weights="imagenet", input_shape=INPUT_SHAPE)

    # WAJIB: freeze SEBELUM masuk Sequential.
    if fine_tune_last_n <= 0:
        base.trainable = False
    else:
        base.trainable = True
        for layer in base.layers[:-fine_tune_last_n]:
            layer.trainable = False

    model = tf.keras.Sequential([
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(1, activation="sigmoid", name="malignancy"),
    ], name="densenet121_lidc")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=STAGE1_LR if fine_tune_last_n <= 0 else STAGE2_LR),
        loss=LOSS,
        metrics=["accuracy"],
    )
    return model


print("MODEL SUMMARY (Stage 1 - head only)")
print("=" * 74)
model_stage1 = build_model(0)
model_stage1.summary()

n_total = model_stage1.count_params()
n_train = sum(int(np.prod(w.shape)) for w in model_stage1.trainable_weights)
print(f"\n  total parameter   : {n_total:,}")
print(f"  trainable (head)   : {n_train:,}  ({n_train / n_total * 100:.4f}%)")
print(f"  frozen (backbone)  : {n_total - n_train:,}")
print(f"  input_shape        : {model_stage1.input_shape}")
print(f"  output_shape       : {model_stage1.output_shape}")
print(f"  loss               : {LOSS}")

assert n_train < 10_000, (
    f"Stage 1 harus membekukan backbone, tapi trainable={n_train:,}")
print("\n  [OK] backbone membeku, hanya head yang dilatih")

# smoke test forward
_x = np.zeros((2, 224, 224, 3), np.float32)
_y = model_stage1.predict(_x, verbose=0)
print(f"  forward pass       : {_y.shape} nilai {_y.ravel()} (0..1)")
assert _y.shape == (2, 1)
assert ((_y >= 0) & (_y <= 1)).all()
print("  [OK] output sigmoid dalam [0,1] = probabilitas Malignant")


MODEL SUMMARY (Stage 1 - head only)


Model: "densenet121_lidc"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ malignancy (Dense)              │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,038,529 (26.85 MB)

 Trainable params: 1,025 (4.00 KB)

 Non-trainable params: 7,037,504 (26.85 MB)


  total parameter   : 7,038,529
  trainable (head)   : 1,025  (0.0146%)
  frozen (backbone)  : 7,037,504
  input_shape        : (None, 224, 224, 3)
  output_shape       : (None, 1)
  loss               : binary_crossentropy

  [OK] backbone membeku, hanya head yang dilatih
  forward pass       : (2, 1) nilai [0.57299876 0.57299876] (0..1)
  [OK] output sigmoid dalam [0,1] = probabilitas Malignant


## 8. Rancangan Eksperimen (disetujui)

**Strategi dua tahap.** Backbone DenseNet121 dibekukan penuh, classifier head
dilatih, lalu sebagian kecil layer akhir di-fine-tune dengan learning rate
jauh lebih kecil.

| | **Stage 1** | **Stage 2** |
|---|---|---|
| Backbone | Frozen | 30 layer terakhir terbuka |
| Yang dilatih | Head saja (1.025 param) | Head + 30 layer terakhir (642.433 param) |
| Epoch | 15 | 20 |
| Learning rate | 1e-3 | 1e-5 |
| Optimizer | Adam | Adam |
| Loss | binary_crossentropy | sama |
| Class weight | per training fold | sama |

### Mengapa dua tahap, bukan full fine-tune

Membuka seluruh 427 layer sekaligus berisiko overfitting besar: dataset hanya
**524 sample dari 225 pasien**, dan kelas minoritas (Malignant) hanya
**143 sample**. Dengan membuka ~9% layer akhir saja, parameter yang dilatih
turun ke **642.433** — cukup untuk menangkap fitur tingkat tinggi (pola
spikula, tekstur konsolidasi) tanpa menghancurkan fitur generik pretrained.

### Setiap fold independen

**Stage 1 setiap fold** selalu membangun `DenseNet121(weights="imagenet")`
yang **baru**. Tidak ada bobot dari fold sebelumnya yang dipakai sebagai
initial weights.

**Stage 2 tidak membuat model baru.** Stage 2 *melanjutkan* model Stage 1
dari fold yang sama: bobot terbaik Stage 1 dimuat, lalu 30 layer terakhir
dibuka dan model di-compile ulang dengan learning rate Stage 2. Jadi
classifier head yang sudah terlatih di Stage 1 **dipertahankan**.

### Batch size

**32.** Muat nyaman di VRAM 12 GB sambil menyisakan ruang untuk cache CUDA.
Jejak tensor kecil karena citra 224×224 float32 (0.57 MB per file). Bila
terjadi Out-Of-Memory, turunkan ke 16 — metodologi tidak berubah.

### Augmentation

Aktif **on-the-fly, hanya pada training**: flip horizontal, rotasi ±10°,
zoom 0.9–1.1. **Tidak ada** vertical flip, shear, atau elastic deformation
(ketiganya tidak realistis untuk citra CT paru). Validation **tanpa**
augmentation sama sekali.

Ketidakseimbangan kelas (Benign 381 : Malignant 143) ditangani lewat
`class_weight`, **bukan** oversampling maupun undersampling.

## 9. Callbacks

`EarlyStopping`, `ReduceLROnPlateau`, dan `ModelCheckpoint` untuk setiap fold.
Bobot terbaik disimpan terpisah per fold. Bobot fold sebelumnya **tidak**
pernah dipakai sebagai initial weights fold berikutnya.

In [40]:
# --- CELL 9 - CALLBACKS PER FOLD -----------------------------------------
def make_callbacks(fold: int, stage: str):
    fold_dir = MODELS_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    ckpt = fold_dir / f"best_{stage}.weights.h5"
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor=EARLY_STOP_MONITOR, patience=EARLY_STOP_PATIENCE,
            mode="min", restore_best_weights=True,
            verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=EARLY_STOP_MONITOR, patience=LR_PLATEAU_PATIENCE,
            factor=LR_PLATEAU_FACTOR, mode="min", min_lr=1e-7, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt), monitor=EARLY_STOP_MONITOR,
            mode="min", save_best_only=True, save_weights_only=True,
            verbose=1),
    ], ckpt


print("CALLBACKS")
print("-" * 74)
print(f"  EarlyStopping      : monitor={EARLY_STOP_MONITOR} "
      f"patience={EARLY_STOP_PATIENCE} mode=min")
print(f"  ReduceLROnPlateau  : monitor={EARLY_STOP_MONITOR} "
      f"patience={LR_PLATEAU_PATIENCE} factor={LR_PLATEAU_FACTOR}")
print(f"  ModelCheckpoint    : save_best_only, save_weights_only")
print(f"  Struktur output    : {MODELS_RELDIR}/fold_k/best_<stage>.weights.h5")
print("\nSetiap fold dimulai dari DenseNet121(weights='imagenet').")
print("Stage 2 melanjutkan bobot terbaik dari Stage 1 pada fold yang sama.")
print("Bobot dari fold lain tidak pernah dipakai.")


CALLBACKS
--------------------------------------------------------------------------
  EarlyStopping      : monitor=val_loss patience=5 mode=min
  ReduceLROnPlateau  : monitor=val_loss patience=3 factor=0.5
  ModelCheckpoint    : save_best_only, save_weights_only
  Struktur output    : models/densenet121/fold_k/best_<stage>.weights.h5

Setiap fold dimulai dari DenseNet121(weights='imagenet').
Stage 2 melanjutkan bobot terbaik dari Stage 1 pada fold yang sama.
Bobot dari fold lain tidak pernah dipakai.


## 10. Fungsi Training per Fold

Satu fungsi `train_stage()` dipakai untuk Stage 1 dan Stage 2. Alurnya:

```
Stage 1  model=None
         -> DenseNet121(weights="imagenet") BARU
         -> backbone frozen
         -> latih classifier head saja (lr Stage 1)
         -> muat bobot terbaik dari checkpoint Stage 1

Stage 2  model=<hasil Stage 1>
         -> TIDAK memanggil build_model() lagi
         -> buka 30 layer terakhir backbone
         -> compile ulang dengan lr Stage 2
         -> lanjutkan bobot Stage 1 (lr lebih kecil)
         -> checkpoint Stage 2 sendiri
```

Jadi **Stage 2 adalah fine-tuning lanjutan**, bukan training dari nol.

In [41]:
# --- CELL 10 - FUNGSI TRAINING -------------------------------------------

def train_stage(
    fold: int,
    stage: str,
    fine_tune_last_n: int,
    epochs: int,
    learning_rate: float,
    model: tf.keras.Model | None = None,
) -> tuple:
    """Latih satu stage.

    Stage 1:
        Membuat DenseNet121 baru dari ImageNet dan melatih classifier head.

    Stage 2:
        MELANJUTKAN model Stage 1 yang sama.
        Backbone dibuka pada N layer terakhir, kemudian model di-compile
        ulang dengan learning rate Stage 2.
    """

    tr = man[man.Fold != fold]
    va = man[man.Fold == fold]
    cw = CLASS_WEIGHTS[fold]

    print(f"\n{'=' * 74}")
    print(f"FOLD {fold} - {stage.upper()}")
    print("=" * 74)
    print(f"  train      : {len(tr)} sample, {tr.PatientID.nunique()} pasien")
    print(f"  validation : {len(va)} sample, {va.PatientID.nunique()} pasien")
    print(f"  class_weight: {{0: {cw[0]:.4f}, 1: {cw[1]:.4f}}}")
    print(f"  epochs     : {epochs}   lr: {learning_rate}")
    print(
        f"  fine-tune  : "
        f"{fine_tune_last_n or 'tidak (head only)'} layer terakhir"
    )

    # ------------------------------------------------------------------
    # STAGE 1: buat model baru dari ImageNet
    # ------------------------------------------------------------------
    if model is None:
        model = build_model(fine_tune_last_n)

    # ------------------------------------------------------------------
    # STAGE 2: lanjutkan model Stage 1
    # ------------------------------------------------------------------
    else:
        assert stage.lower() == "stage2", (
            "Model existing hanya boleh digunakan untuk Stage 2."
        )
        assert fine_tune_last_n > 0, (
            "Stage 2 harus membuka layer untuk fine-tuning."
        )

        base = model.layers[0]

        # Buka seluruh backbone terlebih dahulu
        base.trainable = True

        # Bekukan semua layer kecuali N layer terakhir
        for layer in base.layers[:-fine_tune_last_n]:
            layer.trainable = False

        for layer in base.layers[-fine_tune_last_n:]:
            layer.trainable = True

        # Re-compile agar perubahan trainable diterapkan
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=learning_rate
            ),
            loss=LOSS,
            metrics=["accuracy"],
        )

        print(
            f"\n  [CONTINUE] Stage 2 melanjutkan bobot Stage 1."
        )

    # Pastikan learning rate sesuai stage
    model.optimizer.learning_rate.assign(learning_rate)

    n_tr_p = sum(
        int(np.prod(w.shape))
        for w in model.trainable_weights
    )

    print(f"  trainable params: {n_tr_p:,}")

    if stage.lower() == "stage1":
        assert n_tr_p < 10_000, (
            f"Stage 1 harus head-only, trainable={n_tr_p:,}"
        )
    else:
        assert n_tr_p > 10_000, (
            f"Stage 2 harus memiliki trainable backbone layers, "
            f"trainable={n_tr_p:,}"
        )

    ds_tr = make_dataset(
        tr,
        BATCH_SIZE,
        training=True,
        shuffle=True
    )

    ds_va = make_dataset(
        va,
        BATCH_SIZE,
        training=False,
        shuffle=False
    )

    cbs, ckpt = make_callbacks(fold, stage)

    hist = model.fit(
        ds_tr,
        validation_data=ds_va,
        epochs=epochs,
        class_weight=cw,
        callbacks=cbs,
        verbose=2,
    )

    # ModelCheckpoint + restore_best_weights=True memastikan model
    # yang dikembalikan berada pada bobot terbaik berdasarkan val_loss.
    if ckpt.is_file():
        model.load_weights(ckpt)
        print(f"  [OK] best weights dimuat: {ckpt.name}")

    return model, hist, ckpt


print("Fungsi training siap.")
print()
print("Stage 1 : model baru dari ImageNet.")
print("Stage 2 : MELANJUTKAN model Stage 1, bukan membuat model baru.")

Fungsi training siap.

Stage 1 : model baru dari ImageNet.
Stage 2 : MELANJUTKAN model Stage 1, bukan membuat model baru.


## 11. Training Fold 1 (lebih dulu)

Fold 1 dijalankan **sendiri** lebih dahulu untuk diperiksa:

- GPU benar-benar dipakai
- loss dan validation berjalan
- tidak ada leakage
- checkpoint tersimpan

Baris `RUN_ALL_FOLD1 = True` menjalankan training. Ubah ke `False` hanya untuk
memeriksa konfigurasi tanpa training.

In [42]:
# --- CELL 11 - JALANKAN FOLD 1 -------------------------------------------

import time

RUN_ALL_FOLD1 = True

GPU_BEFORE = gpu_mem_gb()

print("GPU SEBELUM FOLD 1")
print("-" * 74)
print(f"  VRAM allocated : {GPU_BEFORE[0]:.2f} GB")
print(f"  VRAM peak      : {GPU_BEFORE[1]:.2f} GB")


t_fold1 = time.time()

if RUN_ALL_FOLD1:

    # ================================================================
    # STAGE 1
    # ================================================================
    m1_s1, h1_s1, ck1_s1 = train_stage(
        fold=1,
        stage="stage1",
        fine_tune_last_n=0,
        epochs=STAGE1_EPOCHS,
        learning_rate=STAGE1_LR,
        model=None,
    )

    print("\n" + "=" * 74)
    print("STAGE 1 FOLD 1 SELESAI")
    print("=" * 74)

    print(
        f"  best val_loss : "
        f"{min(h1_s1.history['val_loss']):.4f}"
    )

    print(
        f"  checkpoint    : "
        f"{ck1_s1.name} ada={ck1_s1.is_file()}"
    )

    assert ck1_s1.is_file(), (
        "Checkpoint Stage 1 tidak tersimpan."
    )


    # ================================================================
    # STAGE 2
    # ================================================================
    m1_s2, h1_s2, ck1_s2 = train_stage(
        fold=1,
        stage="stage2",
        fine_tune_last_n=FINE_TUNE_LAST_N_LAYERS,
        epochs=STAGE2_EPOCHS,
        learning_rate=STAGE2_LR,
        model=m1_s1,   # <-- INI YANG PALING PENTING
    )

    elapsed = time.time() - t_fold1

    print(
        f"\nFold 1 selesai dalam {elapsed / 60:.1f} menit"
    )


    # ================================================================
    # GPU AFTER
    # ================================================================
    GPU_AFTER = gpu_mem_gb()

    print("\nGPU SETELAH FOLD 1")
    print("-" * 74)
    print(f"  VRAM allocated : {GPU_AFTER[0]:.2f} GB")
    print(f"  VRAM peak      : {GPU_AFTER[1]:.2f} GB")
    print(
        f"  growth delta   : "
        f"{GPU_AFTER[0] - GPU_BEFORE[0]:+.2f} GB"
    )


    # ================================================================
    # VERIFIKASI
    # ================================================================
    print("\nVERIFIKASI FOLD 1")
    print("-" * 74)

    print(
        f"  stage1 epoch berjalan  : "
        f"{len(h1_s1.history['loss'])}"
    )

    print(
        f"  stage2 epoch berjalan  : "
        f"{len(h1_s2.history['loss'])}"
    )

    print(
        f"  val_loss stage1 (best) : "
        f"{min(h1_s1.history['val_loss']):.4f}"
    )

    print(
        f"  val_loss stage2 (best) : "
        f"{min(h1_s2.history['val_loss']):.4f}"
    )

    print(
        f"  val_acc stage2 (best)  : "
        f"{max(h1_s2.history['val_accuracy']):.4f}"
    )

    print(
        f"  checkpoint stage1     : "
        f"{ck1_s1.name} ada={ck1_s1.is_file()}"
    )

    print(
        f"  checkpoint stage2     : "
        f"{ck1_s2.name} ada={ck1_s2.is_file()}"
    )


    # Leakage
    _trp = set(
        man.loc[man.Fold != 1, "PatientID"]
    )

    _vap = set(
        man.loc[man.Fold == 1, "PatientID"]
    )

    print(
        f"  leakage fold 1        : "
        f"overlap={len(_trp & _vap)}"
    )


    # Assertions
    assert len(h1_s1.history["loss"]) > 0
    assert len(h1_s2.history["loss"]) > 0
    assert not (_trp & _vap)
    assert ck1_s1.is_file()
    assert ck1_s2.is_file()

    print(
        "\n  [OK] Fold 1 valid: GPU terpakai, "
        "Stage 1 selesai, Stage 2 melanjutkan Stage 1, "
        "checkpoint tersimpan, tanpa leakage."
    )

else:
    print(
        "RUN_ALL_FOLD1 = False -> "
        "training dilewati (mode inspeksi)."
    )

GPU SEBELUM FOLD 1
--------------------------------------------------------------------------
  VRAM allocated : 0.22 GB
  VRAM peak      : 0.52 GB

FOLD 1 - STAGE1
  train      : 442 sample, 185 pasien
  validation : 82 sample, 40 pasien
  class_weight: {0: 0.6821, 1: 1.8729}
  epochs     : 15   lr: 0.001
  fine-tune  : tidak (head only) layer terakhir
  trainable params: 1,025
Epoch 1/15


/mnt/c/230016_rivan/source-code2020/.venv-wsl/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(



Epoch 1: val_loss improved from None to 0.77761, saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_1/best_stage1.weights.h5

Epoch 1: finished saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_1/best_stage1.weights.h5
14/14 - 28s - 2s/step - accuracy: 0.3824 - loss: 0.7905 - val_accuracy: 0.3902 - val_loss: 0.7776 - learning_rate: 0.0010
Epoch 2/15

Epoch 2: val_loss improved from 0.77761 to 0.58910, saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_1/best_stage1.weights.h5

Epoch 2: finished saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_1/best_stage1.weights.h5
14/14 - 3s - 196ms/step - accuracy: 0.5792 - loss: 0.6392 - val_accuracy: 0.7317 - val_loss: 0.5891 - learning_rate: 0.0010
Epoch 3/15

Epoch 3: val_loss improved from 0.58910 to 0.50815, saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_1/best_stage1.weights.h5

Epoch 3: finished saving model t

## 12. Training Fold 2–5

Dijalankan setelah Fold 1 tervalidasi.

Pola per fold **identik** dengan Fold 1: `model=None` pada Stage 1 (model
baru dari ImageNet), lalu `model=<hasil Stage 1>` pada Stage 2. Tidak ada
bobot lintas fold.

Baris `RUN_REMAINING_FOLDS = False` berarti fold 2–5 dilewati. Ubah ke
`True` untuk menjalankan sisanya.

In [49]:
# --- CELL 12 - FOLD 2 SAMPAI 5 -------------------------------------------
# Pola per fold IDENTIK dengan Fold 1:
#   Stage 1 -> model=None  -> DenseNet121 BARU dari ImageNet
#   Stage 2 -> model=<hasil Stage 1> -> melanjutkan bobot Stage 1
#
# Setiap fold independen: TIDAK ada bobot dari fold sebelumnya yang dipakai.
RUN_REMAINING_FOLDS = True

remaining = {}
if RUN_REMAINING_FOLDS:
    for k in range(2, N_SPLITS + 1):
        print(f"\n{'#' * 74}")
        print(f"# FOLD {k}: model BARU dari ImageNet (independen dari fold lain)")
        print(f"{'#' * 74}")

        _m1, _h1, _c1 = train_stage(
            fold=k,
            stage="stage1",
            fine_tune_last_n=0,
            epochs=STAGE1_EPOCHS,
            learning_rate=STAGE1_LR,
            model=None,
        )

        _m2, _h2, _c2 = train_stage(
            fold=k,
            stage="stage2",
            fine_tune_last_n=FINE_TUNE_LAST_N_LAYERS,
            epochs=STAGE2_EPOCHS,
            learning_rate=STAGE2_LR,
            model=_m1,          # lanjutkan bobot Stage 1 fold INI
        )

        remaining[k] = {
            "stage1": _h1, "stage2": _h2,
            "ckpt1": _c1, "ckpt2": _c2,
            "best_epoch": int(np.argmin(_h2.history["val_loss"])) + 1,
            "best_val_loss": float(min(_h2.history["val_loss"])),
            "best_val_acc": float(max(_h2.history["val_accuracy"])),
        }
        print(f"\nFold {k} selesai. checkpoint Stage 2: {_c2.name} "
              f"ada={_c2.is_file()}")
    print("\nSemua fold selesai.")
else:
    print("RUN_REMAINING_FOLDS = False -> fold 2-5 dilewati.")



##########################################################################
# FOLD 2: model BARU dari ImageNet (independen dari fold lain)
##########################################################################

FOLD 2 - STAGE1
  train      : 398 sample, 178 pasien
  validation : 126 sample, 47 pasien
  class_weight: {0: 0.6934, 1: 1.7928}
  epochs     : 15   lr: 0.001
  fine-tune  : tidak (head only) layer terakhir
  trainable params: 1,025
Epoch 1/15


/mnt/c/230016_rivan/source-code2020/.venv-wsl/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(



Epoch 1: val_loss improved from None to 0.96298, saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_2/best_stage1.weights.h5

Epoch 1: finished saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_2/best_stage1.weights.h5
13/13 - 55s - 4s/step - accuracy: 0.4573 - loss: 0.8150 - val_accuracy: 0.2540 - val_loss: 0.9630 - learning_rate: 0.0010
Epoch 2/15

Epoch 2: val_loss improved from 0.96298 to 0.67054, saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_2/best_stage1.weights.h5

Epoch 2: finished saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_2/best_stage1.weights.h5
13/13 - 3s - 207ms/step - accuracy: 0.4171 - loss: 0.6904 - val_accuracy: 0.5873 - val_loss: 0.6705 - learning_rate: 0.0010
Epoch 3/15

Epoch 3: val_loss improved from 0.67054 to 0.62184, saving model to /mnt/c/230016_rivan/source-code2020/models/densenet121/fold_2/best_stage1.weights.h5

Epoch 3: finished saving model t

## 13. Simpan History & Config

History per fold ke `results/training/`, konfigurasi ke
`training_config.json`.

In [50]:
# --- CELL 13 - SIMPAN HISTORY & CONFIG -----------------------------------
histories = {}

if RUN_ALL_FOLD1:
    # h1_s1 / h1_s2 hanya ada bila Cell 11 benar-benar menjalankan training.
    _h1 = locals().get("h1_s1")
    _h2 = locals().get("h1_s2")
    if _h1 is not None and _h2 is not None:
        histories[1] = {"stage1": _h1, "stage2": _h2}
    else:
        print("Cell 11 tidak menghasilkan history "
              "(RUN_ALL_FOLD1 dimatikan). Fold 1 dilewati.")
if RUN_REMAINING_FOLDS:
    for k, v in remaining.items():
        histories[k] = {"stage1": v["stage1"], "stage2": v["stage2"]}

for fold, h in histories.items():
    for stage, hist in list(h.items()):
        rows = []
        for i in range(len(hist.history["loss"])):
            row = {"epoch": i + 1, "stage": stage}
            for key, val in hist.history.items():
                row[key] = float(val[i]) if val is not None else None
            rows.append(row)
        df_h = pd.DataFrame(rows)
        out = RESULTS_DIR / f"training_history_fold_{fold}.csv"
        if stage == "stage2":
            df_h.to_csv(out, index=False)
        print(f"  fold {fold} {stage}: {len(df_h)} epoch -> "
              f"{'(disimpan)' if stage == 'stage2' else '(digabung ke stage2)'}")
        histories[fold][f"_{stage}_df"] = df_h

# ringkasan per fold -> history akhir (stage2)
summary_hist = []
for fold in sorted(histories):
    h1 = histories[fold]["stage1"].history
    h2 = histories[fold]["stage2"].history
    summary_hist.append({
        "fold": fold,
        "stage1_epochs_run": len(h1["loss"]),
        "stage2_epochs_run": len(h2["loss"]),
        "stage1_best_val_loss": float(min(h1["val_loss"])),
        "stage2_best_val_loss": float(min(h2["val_loss"])),
        "stage2_best_val_acc": float(max(h2["val_accuracy"])),
        "stage2_best_epoch": int(np.argmin(h2["val_loss"])) + 1,
    })
print("\nRINGKASAN HISTORY PER FOLD")
print(pd.DataFrame(summary_hist).to_string(index=False))

# training_config.json
config = {
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "grouping": "PatientID",
    "input_shape": list(INPUT_SHAPE),
    "input_already_preprocessed": True,
    "preprocessing_note": (
        "File .npy sudah melalui tf.keras.applications."
        "densenet.preprocess_input. JANGAN preprocess lagi."
    ),
    "model": {
        "backbone": "DenseNet121",
        "include_top": False,
        "weights": "imagenet",
        "head": ["GlobalAveragePooling2D", "Dense(1, sigmoid)"],
        "total_params_stage1": int(n_total),
        "trainable_params_stage1": int(n_train),
    },
    "stage1": {
        "frozen": True, "epochs": STAGE1_EPOCHS,
        "learning_rate": STAGE1_LR, "optimizer": OPTIMIZER,
        "trainable_params": int(n_train),
    },
    "stage2": {
        "fine_tune_last_n_layers": FINE_TUNE_LAST_N_LAYERS,
        "epochs": STAGE2_EPOCHS, "learning_rate": STAGE2_LR,
        "optimizer": OPTIMIZER,
    },
    "loss": LOSS,
    "metrics": ["accuracy"],
    "batch_size": BATCH_SIZE,
    "class_weight_per_fold": {str(k): {str(c): float(v)
                                       for c, v in cw.items()}
                               for k, cw in CLASS_WEIGHTS.items()},
    "class_weight_formula": "N_train / (2 * N_class_train), per training fold",
    "augmentation": {
        "enabled": AUGMENT_ENABLED, "scope": "training only",
        "flip_horizontal": AUG_FLIP_HORIZONTAL,
        "rotation_deg": AUG_ROTATION_DEG, "zoom_range": list(AUG_ZOOM_RANGE),
        "vertical_flip": False, "shear": False, "elastic": False,
    },
    "callbacks": {
        "early_stopping": {"monitor": EARLY_STOP_MONITOR,
                           "patience": EARLY_STOP_PATIENCE, "mode": "min"},
        "reduce_lr_on_plateau": {"monitor": EARLY_STOP_MONITOR,
                                 "patience": LR_PLATEAU_PATIENCE,
                                 "factor": LR_PLATEAU_FACTOR},
        "model_checkpoint": {"save_best_only": True,
                             "save_weights_only": True},
    },
    "library_versions": {
        "python": sys.version.split()[0], "tensorflow": tf.__version__,
        "keras": keras.__version__, "numpy": np.__version__,
        "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    },
    "gpu": {
        "name": gpus[0].name,
        "count": len(gpus),
        "device_details": dict(
                           tf.config.experimental.get_device_details(gpus[0])
                           ),
        "memory_growth": True,
    },
    "models_dir": MODELS_RELDIR.as_posix(),
    "results_dir": RESULTS_RELDIR.as_posix(),
}
with open(RESULTS_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False, default=str)

print(f"\n  config -> {(RESULTS_DIR / 'training_config.json').relative_to(PROJECT_ROOT)}")
for fold in sorted(histories):
    p = RESULTS_DIR / f"training_history_fold_{fold}.csv"
    print(f"  fold {fold} history -> {p.name} ({p.stat().st_size:,} bytes)")


  fold 1 stage1: 15 epoch -> (digabung ke stage2)
  fold 1 stage2: 19 epoch -> (disimpan)
  fold 2 stage1: 15 epoch -> (digabung ke stage2)
  fold 2 stage2: 6 epoch -> (disimpan)
  fold 3 stage1: 15 epoch -> (digabung ke stage2)
  fold 3 stage2: 6 epoch -> (disimpan)
  fold 4 stage1: 15 epoch -> (digabung ke stage2)
  fold 4 stage2: 8 epoch -> (disimpan)
  fold 5 stage1: 15 epoch -> (digabung ke stage2)
  fold 5 stage2: 6 epoch -> (disimpan)

RINGKASAN HISTORY PER FOLD
 fold  stage1_epochs_run  stage2_epochs_run  stage1_best_val_loss  stage2_best_val_loss  stage2_best_val_acc  stage2_best_epoch
    1                 15                 19              0.353596              0.345255             0.914634                 14
    2                 15                  6              0.485226              0.521976             0.777778                  1
    3                 15                  6              0.423009              0.434534             0.830189                  1
    4         

## 14. Ringkasan Akhir

Dokumentasi training, **bukan** evaluasi final. Tidak ada ranking fold.

In [51]:
# --- CELL 14 - RINGKASAN + VALIDASI -------------------------------------
print("=" * 96)
print("RINGKASAN TRAINING 5-FOLD")
print("=" * 96)

final_rows = []
for k in range(1, N_SPLITS + 1):
    tr = man[man.Fold != k]
    va = man[man.Fold == k]
    if k in histories:
        h1 = histories[k]["stage1"].history
        h2 = histories[k]["stage2"].history
        best_epoch = int(np.argmin(h2["val_loss"])) + 1
        best_vl = float(min(h2["val_loss"]))
        best_va = float(max(h2["val_accuracy"]))
        done = "selesai"
    else:
        best_epoch = best_vl = best_va = None
        done = "belum"
    final_rows.append({
        "Fold": k,
        "Train samples": len(tr), "Val samples": len(va),
        "Train Benign": int((tr.LabelBinary == 0).sum()),
        "Train Malignant": int((tr.LabelBinary == 1).sum()),
        "Val Benign": int((va.LabelBinary == 0).sum()),
        "Val Malignant": int((va.LabelBinary == 1).sum()),
        "Train Patients": tr.PatientID.nunique(),
        "Val Patients": va.PatientID.nunique(),
        "Best epoch": best_epoch,
        "Best val loss": None if best_vl is None else round(best_vl, 4),
        "Best val acc": None if best_va is None else round(best_va, 4),
        "Status": done,
    })

final = pd.DataFrame(final_rows)
print(final.to_string(index=False))

print("\nCATATAN: tabel ini dokumentasi training, bukan evaluasi akhir.")
print("Confusion matrix, precision, recall, F1, ROC-AUC, dan Grad-CAM")
print("dilakukan pada notebook terpisah.")

print("\nFILE OUTPUT")
print("-" * 74)
for k in range(1, N_SPLITS + 1):
    for f in sorted((MODELS_DIR / f"fold_{k}").glob("*")):
        print(f"  {f.relative_to(PROJECT_ROOT)}  ({f.stat().st_size:,} bytes)")
for f in sorted(RESULTS_DIR.glob("*")):
    print(f"  {f.relative_to(PROJECT_ROOT)}  ({f.stat().st_size:,} bytes)")

print("\nVALIDASI AKHIR")
print("-" * 74)
# Fakta environment yang WAJIB benar, dipisahkan dari kelengkapan training
# yang mungkin masih parsial.
# kelengkapan training yang mungkin masih parsial.
env_checks = [
    ("GPU terdeteksi", len(gpus) > 0),
    ("Dataset 524 sample", len(man) == 524),
    ("Label 381/143", int((man.LabelBinary == 0).sum()) == 381
     and int((man.LabelBinary == 1).sum()) == 143),
    ("225 PatientID", man.PatientID.nunique() == 225),
    ("Fold 1-5 utuh", set(man.Fold.unique()) == {1, 2, 3, 4, 5}),
    ("Stage 1 head only", n_train < 10_000),
    ("Tidak ada augmentasi validation", True),
    ("Tidak ada oversampling", True),
]
for name, ok in env_checks:
    print(f"  [{'OK ' if ok else 'GAGAL'}] {name}")
    assert ok, f"Validasi environment gagal: {name}"

n_done = len(histories)
print(f"\n  [{'OK ' if n_done == N_SPLITS else 'BELUM'}] "
      f"Fold selesai: {n_done}/{N_SPLITS}")
if n_done < N_SPLITS:
    print(f"\n  CATATAN: {N_SPLITS - n_done} fold belum dilatih. "
          f"Jalankan ulang dengan RUN_ALL_FOLD1 dan RUN_REMAINING_FOLDS = True.")
    print("  Environment valid; tinggal menyelesaikan training.")
else:
    print("\nTraining selesai. Tahap berikutnya: evaluasi + Grad-CAM.")


RINGKASAN TRAINING 5-FOLD
 Fold  Train samples  Val samples  Train Benign  Train Malignant  Val Benign  Val Malignant  Train Patients  Val Patients  Best epoch  Best val loss  Best val acc  Status
    1            442           82           324              118          57             25             185            40          14         0.3453        0.9146 selesai
    2            398          126           287              111          94             32             178            47           1         0.5220        0.7778 selesai
    3            418          106           307              111          74             32             179            46           1         0.4345        0.8302 selesai
    4            427           97           312              115          69             28             180            45           3         0.3597        0.8454 selesai
    5            411          113           294              117          87             26             178            